# L94 · Aurora v1 全景 Demo — 综合展示所有能力，面试材料与证据链整理

**目标**：打磨一个可展示的 Demo，整理 Aurora 的证据链，准备面试材料。

🔗 **Aurora 连接**：
- `src/aurora/audio/` — Audio Core（FFT / STFT / Mel / MFCC，全手写）
- `src/aurora/train/` — 训练循环与模型权重
- `src/aurora/asr/`   — 语音识别模块
- `src/aurora/rec/`   — 推荐引擎
- `src/aurora/agent/` — Agent 调度层
- `tests/`            — 每个模块的基准测试（针对 numpy / 公式验证）


## 为什么需要"交付意识"

一个研究系统的价值，不仅取决于它能做什么，还取决于别人能不能在 5 分钟内看懂它能做什么。
Aurora v1 的核心卖点是**从零实现**（no API wrappers）：面试官问"你怎么做 FFT"，答案不是"用 librosa"，
而是打开一个 notebook 直接运行。本节把 6 个月的成果组织成一条清晰的**证据链**，
并把演示流程、面试问答、简历 bullet 全部固化下来。


In [ ]:
import numpy as np


## 1. 证据链（Evidence Chain）

"我从零实现了 FFT"——这句话需要**可验证**的证据支撑，否则面试官无法区分你和背面经的人。
Aurora 的证据链由四层组成：

| 层级 | 形式 | 作用 |
|------|------|------|
| **GitHub history** | commit log + PR | 证明过程，不只是结果 |
| **博客文章** | `docs/blog/` Markdown | 证明理解深度，能讲清楚 |
| **测试结果** | `pytest tests/` 通过 | 证明代码正确，针对公式验证 |
| **Demo 录屏** | GIF / MP4 | 证明系统端到端可跑 |

每一层的核心逻辑：**不可伪造性**递增。commit 时间戳不可改，测试对公式有误差界，
录屏显示真实推理时延。面试材料要按这四层准备，缺哪层就补哪层。


In [ ]:
# 证据链自检：打印各层的关键文件路径，确认它们存在
import os

AURORA_ROOT = os.path.expanduser("~/AURORA")

evidence = {
    "GitHub history": ".git/",
    "博客文章":       "docs/blog/",
    "测试目录":       "tests/",
    "ADR 决策记录":   "docs/adr/",
}

print("── 证据链自检 ──")
for label, rel in evidence.items():
    full = os.path.join(AURORA_ROOT, rel)
    exists = os.path.exists(full)
    status = "✅" if exists else "❌ 缺失"
    print(f"  {status}  {label:12s}  {full}")


## 2. 如何 Present AI 系统：WWHR 框架

面试官在 5 分钟内要判断：这个人能不能在团队里独立推进一个 AI 系统？
结构化表达比堆细节有效得多。推荐用 **WWHR**（What / Why / How / Results / Limits）：

```
What    — 系统做什么，一句话
Why     — 为什么从头实现而不用 API（研究诚信 + 面试场景）
How     — 关键算法，能展开讲（FFT → STFT → Mel → MFCC → 模型）
Results — 可量化指标（WER、延迟、测试通过率）
Limits  — 诚实说明边界（单说话人、小词表、非实时）
```

**陷阱**：把大量时间花在 What 上，跳过 Why 和 Limits。面试官最关心 Why（判断思维深度）
和 Limits（判断自我认知）。Results 提供可信度锚点，但没有 Why 的 Results 是空的。


In [ ]:
# WWHR 模板：打印 Aurora v1 的标准介绍稿（可直接背诵或改写）
wwhr = {
    "What": (
        "Aurora 是一个从零实现的音频 AI 系统，"
        "覆盖特征提取（MFCC）、语音识别（ASR）、推荐和 Agent 四个模块。"
    ),
    "Why": (
        "所有核心算法（FFT、Mel 滤波器组、DCT）全部手写，不调用 librosa / scipy.signal。"
        "目的是证明对 DSP 数学的第一性原理理解，而非 API 调用能力。"
    ),
    "How": (
        "FFT 用 Cooley-Tukey 蝶形分解实现，O(N log N)；"
        "Mel 滤波器组按 HTK 公式手动构造三角滤波器；"
        "MFCC 在 Mel 能量上做手写 DCT-II；"
        "训练循环用纯 NumPy（或轻量 PyTorch）实现梯度下降。"
    ),
    "Results": (
        "所有模块通过与 numpy.fft / 参考公式的误差测试（< 1e-10）；"
        "端到端 ASR demo 在 LibriSpeech clean 子集 WER < X%（待填写实测值）。"
    ),
    "Limits": (
        "当前仅支持单说话人、16kHz 采样；"
        "推理非实时（无流式解码）；"
        "模型未在大规模数据上调优，WER 高于工业系统。"
    ),
}

print("── Aurora v1 WWHR 介绍稿 ──\n")
for key, val in wwhr.items():
    print(f"[{key}]\n  {val}\n")


## 3. 简历摘录：每个月份 → 一个 Bullet

简历上的 AI 系统 bullet 格式：

```
[方法] 实现 [功能]，[指标/对比]
```

例子：

```
• 手写 Cooley-Tukey FFT（O(N log N)），误差 < 1e-10（对比 numpy.fft）
• 构造 80-channel Mel 滤波器组 + DCT-II，端到端 MFCC 流水线无第三方 DSP 库
• 训练轻量 ASR 模型，LibriSpeech clean WER < X%，推理延迟 < Y ms（CPU）
• 实现基于 MFCC 相似度的音频推荐引擎，top-5 精度 Z%
• 搭建 Agent 层，自动调度特征提取 → 推理 → 结果聚合流水线
```

每条 bullet 的写法要点：
- **动词精确**：「手写」「构造」「实现」「搭建」，不用「使用」「基于」
- **指标可查**：误差界来自测试日志，WER 来自评测脚本
- **对比锚点**：有对比（vs numpy.fft、vs librosa）才有说服力


In [ ]:
# 简历 bullet 生成器：按月份输出当前已完成的 bullet
bullets = [
    ("M1", "Audio Core",  "手写 Cooley-Tukey FFT（O(N log N)），数值误差 < 1e-10（对比 numpy.fft）"),
    ("M2", "STFT / Mel",  "实现 STFT + 80-channel Mel 滤波器组（HTK 公式），无 librosa / scipy.signal"),
    ("M3", "MFCC",        "构造手写 DCT-II，完整 MFCC 流水线通过公式基准测试"),
    ("M4", "训练循环",    "从零搭建训练循环，损失曲线收敛，模型权重可复现"),
    ("M5", "ASR",         "端到端语音识别模块，输入音频 → 文本，WER 待填写实测值"),
    ("M6", "Agent/Demo",  "Agent 调度层 + v1 Demo，覆盖特征提取→推理→推荐全链路"),
]

print("── 简历 Bullet（按月） ──\n")
for month, module, text in bullets:
    print(f"  [{month}] {module}")
    print(f"    • {text}")
    print()

# 检查：bullet 数量应等于里程碑数量
assert len(bullets) == 6, f"期望 6 条 bullet，实际 {len(bullets)}"
print("✅ 6 条 bullet 均已填写")


## 参数实验：5 分钟演示排练 + 面试 Q&A

### 5 分钟演示流程

| 时间 | 内容 | 关键操作 |
|------|------|---------|
| 0:00–0:30 | 一句话介绍 Aurora | 说 WWHR 中的 What + Why |
| 0:30–1:30 | 运行 FFT notebook  | 跑 `week01/day4_euler.ipynb`，展示蝶形图 |
| 1:30–2:30 | 运行 MFCC 流水线   | 跑 `month03/` notebook，展示 MFCC 热图 |
| 2:30–3:30 | 运行 ASR Demo      | 跑 `month05/` notebook，展示文本输出 |
| 3:30–4:30 | 展示测试结果       | `pytest tests/ -v`，全绿 |
| 4:30–5:00 | Limits + Q&A       | 主动说限制，邀请追问 |

**预备面试问题 10 个**（附答案方向，详见下方 code 格）：
1. 为什么不用 librosa？
2. FFT 的时间复杂度是多少，你怎么验证的？
3. Mel 滤波器组的频率间距为什么不是线性的？
4. DCT-II 和 DFT 有什么区别？
5. MFCC 的第 0 个系数代表什么？
6. 你的模型在噪声环境下表现如何？
7. 如何扩展到流式（实时）推理？
8. 推荐模块用的是什么相似度度量？
9. Agent 层怎么处理失败重试？
10. 如果给你三个月，你最优先改进什么？


In [ ]:
# 面试 Q&A 打印：运行一遍，确认答案都在脑子里
qa = [
    (
        "为什么不用 librosa？",
        "librosa.stft / librosa.feature.mfcc 是黑盒——你无法解释内部实现。"
        "手写逼着你理解每一步数学；面试时能展开讲，不是只能说'调了个函数'。"
    ),
    (
        "FFT 的时间复杂度是多少，你怎么验证的？",
        "O(N log N)，Cooley-Tukey 每级做 N/2 次蝶形，共 log₂N 级。"
        "验证方式：对不同 N 计时，log-log 图斜率应约等于 1（log N 项），"
        "同时与 numpy.fft 比对数值误差 < 1e-10。"
    ),
    (
        "Mel 滤波器组的频率间距为什么不是线性的？",
        "人耳对低频分辨率高、对高频分辨率低，Mel 尺度模拟这一特性。"
        "公式 `mel = 2595 * log10(1 + hz/700)`，低频段 Mel 间距小、Hz 间距也小，"
        "所以滤波器在低频密集、高频稀疏。"
    ),
    (
        "DCT-II 和 DFT 有什么区别？",
        "DFT 假设信号周期延拓，会在边界产生不连续；DCT-II 假设偶对称延拓，"
        "边界连续，能量更集中在低频系数，适合压缩和特征提取。"
        "MFCC 用 DCT-II 是因为 Mel 能量之间高度相关，DCT 去相关后系数更独立。"
    ),
    (
        "MFCC 的第 0 个系数代表什么？",
        "C0 是所有 Mel 频带能量的加权和，近似于对数总能量（音量）。"
        "实践中常被丢弃或替换为对数帧能量，因为它对信道差异敏感。"
    ),
    (
        "你的模型在噪声环境下表现如何？",
        "当前未做噪声增强，在 SNR < 10dB 时 WER 显著上升。"
        "改进方向：数据增强（加噪/混响）或前端谱减法。这是已知 Limit。"
    ),
    (
        "如何扩展到流式（实时）推理？",
        "STFT 需要改成滑动窗口、每帧增量处理；模型换成流式解码（CTC greedy 或 beam）。"
        "当前的批量推理框架不支持，需要重构推理入口。"
    ),
    (
        "推荐模块用的是什么相似度度量？",
        "MFCC 向量之间的余弦相似度。公式：`cos(a,b) = dot(a,b) / (||a|| * ||b||)`。"
        "候选方向：用 DTW 做时序对齐，或加 FAISS 做近似最近邻加速。"
    ),
    (
        "Agent 层怎么处理失败重试？",
        "当前是简单的线性 pipeline，无重试逻辑——这是已知 Limit。"
        "生产级做法：每个步骤包装成 Task，失败后指数退避重试，超过阈值告警。"
    ),
    (
        "如果给你三个月，你最优先改进什么？",
        "流式推理 > 噪声鲁棒性 > 大词表语言模型融合。"
        "流式是最大工程缺口，噪声鲁棒性对实用场景影响最直接，"
        "语言模型融合（shallow fusion）能快速拉低 WER。"
    ),
]

print("── 面试 Q&A 速查 ──\n")
for i, (q, a) in enumerate(qa, 1):
    print(f"Q{i:02d}: {q}")
    print(f"  A:  {a}")
    print()

assert len(qa) == 10
print("✅ 10 道题全部准备完毕")


## 本课收束

本节没有新的编码任务，核心输出是**三件可交付物**：证据链自检报告、WWHR 介绍稿、简历 bullet 清单。
整个 Aurora 系统以 `src/aurora/audio/` 为基础、`tests/` 为验证层，形成从 FFT 到 Agent 的完整证据链。
面试时的核心竞争力不在于指标高低，而在于能逐层展开 Why / How / Limits，这三个函数名比任何 WER 数字都更有说服力。
下一步：录制 Demo 录屏、更新 `docs/blog/` 里的 M6 博客文章，把 Aurora v1 打包成可分享的 GitHub Release。
